# Stage1 0bp 差区间和固定状态归因

假设：冻结 Stage1 在 SOXX 5/20 日均跌的决策前状态及部分季度持续落后。此卡只读已归档 Qlib 原生账户和已固定状态，不拟合、不回测、不挑窗口；分组统计不构成可交易策略。来源、预算和停止条件见 `plan.json`。

In [1]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
from src.analysis.regime_failure import regime_features

root=Path.cwd()
card=root/'research/stage1_0bp_regime_attribution_20261022'
plan=json.loads((card/'plan.json').read_text())
for key in ('source_stage1','source_benchmark'):
    source=plan[key]; summary=json.loads((root/source['card']/'summary.json').read_text())
    artifact_root=Path(summary['recorder_local_dir'])/'artifacts'
    identities=source['artifacts'] if 'artifacts' in source else {source['artifact']:source['sha256']}
    for name,sha in identities.items():
        assert hashlib.sha256((artifact_root/name).read_bytes()).hexdigest()==sha, (key,name)
assert sum(len(plan[k].get('artifacts',{plan[k].get('artifact'):None})) for k in ('source_stage1','source_benchmark'))==6
sa=json.loads((root/plan['source_stage1']['card']/'summary.json').read_text())
sb=json.loads((root/plan['source_benchmark']['card']/'summary.json').read_text())
qlib.init(provider_uri=sa['provider_uri'],region='us')
with R.uri_context(sa['recorder_uri']):
    rec=R.get_recorder(recorder_id=sa['recorder_id'],experiment_id=sa['experiment_id'])
    frame=rec.load_object('inputs/frame')
    stage1=rec.load_object('paths/A_active_research_replay_0')['replay']
    source_year=rec.load_object('analysis/annual_metrics')
    source_quarter=rec.load_object('analysis/quarterly_metrics')
    source_rolling=rec.load_object('analysis/rolling/A_active_research_replay_0')
with R.uri_context(sb['recorder_uri']):
    rec=R.get_recorder(recorder_id=sb['recorder_id'],experiment_id=sb['experiment_id'])
    benchmark=rec.load_object('native/raw_SOXX_0')['authority']
assert len(frame)==len(stage1)==len(benchmark)==944
assert stage1.trade_date.astype(str).equals(benchmark.trade_date.astype(str))
assert stage1.trade_date.astype(str).equals(frame.trade_date.astype(str))
assert (frame.feature_available_at < frame.decision_at).all()
print('Source SHA, native account identities, decision clock and 944 calendars verified.')

In [2]:
states=regime_features(frame)
daily=pd.DataFrame({
    'trade_date':pd.to_datetime(stage1.trade_date),
    'signal_date':pd.to_datetime(stage1.signal_date),
    'regime':states.regime.to_numpy(),
    'relative_log':np.log1p(stage1.strategy_return.to_numpy(float))-np.log1p(benchmark.strategy_return.to_numpy(float)),
    'filled_cash_weight':stage1.ending_cash_weight.to_numpy(float),
    'filled_soxl_weight':stage1.ending_soxl_weight.to_numpy(float),
    'risk_nonzero':stage1.risk_state.ne(0).to_numpy(),
    'open_execute':stage1.execute_open_target.to_numpy(bool),
})
daily['year']=daily.trade_date.dt.to_period('Y').astype(str)
daily['quarter']=daily.trade_date.dt.to_period('Q').astype(str)
def describe(group):
    out=group.agg(sessions=('relative_log','size'),relative_log=('relative_log','sum'),
                  mean_filled_cash=('filled_cash_weight','mean'),mean_filled_soxl=('filled_soxl_weight','mean'),
                  risk_nonzero_fraction=('risk_nonzero','mean'),open_execute_fraction=('open_execute','mean'))
    out['relative_wealth_change_pct']=100*np.expm1(out.relative_log)
    return out.reset_index()
annual=describe(daily.groupby('year',sort=True))
quarters=describe(daily.groupby('quarter',sort=True))
regimes=describe(daily.groupby('regime',sort=True))
fixed_regime_order=['uptrend','rebound','pullback','decline']
regimes['regime']=pd.Categorical(regimes.regime,categories=fixed_regime_order,ordered=True)
regimes=regimes.sort_values('regime').reset_index(drop=True)
source_year=source_year[(source_year.family=='A_active_research_replay')&(source_year.slippage_bps==0)].set_index('period')
source_quarter=source_quarter[(source_quarter.family=='A_active_research_replay')&(source_quarter.slippage_bps==0)].set_index('period')
np.testing.assert_allclose(annual.set_index('year').relative_log.loc[source_year.index],source_year.log_excess_sum,atol=1e-12)
np.testing.assert_allclose(quarters.set_index('quarter').relative_log.loc[source_quarter.index],source_quarter.log_excess_sum,atol=1e-12)
computed_rolling=daily.relative_log.rolling(126).sum().iloc[125:].reset_index(drop=True)
np.testing.assert_allclose(computed_rolling,source_rolling.relative_log,atol=1e-12)
assert len(computed_rolling)==819
ratio=float(np.exp(daily.relative_log.sum()))
assert abs(ratio-0.6860064369263443)<1e-10
assert abs(float(computed_rolling.gt(0).mean())-0.31135531135531136)<1e-12
assert regimes.sessions.sum()==944
print('0bp relative wealth',ratio,'rolling positive',float(computed_rolling.gt(0).mean()))
display(regimes,quarters)

In [3]:
metrics=pd.read_csv(root/'research/stage1_domain_heads_20261018/metrics.csv')
base=metrics[(metrics.family=='A_active_research_replay')&(metrics.slippage_bps==0)].iloc[0]
assert abs(base.relative_wealth-ratio)<1e-12
assert abs(base.mean_ending_cash_weight-daily.filled_cash_weight.mean())<1e-12
complete=source_year.loc[source_year.complete_period.astype(bool)]
assert len(complete)==2
summary={
 'experiment':plan['experiment'],'plan_sha256':hashlib.sha256((card/'plan.json').read_bytes()).hexdigest(),
 'decision':'diagnostic_only_Stage1_0bp_not_promoted','objective_achieved':False,
 'new_fits':0,'new_native_accounts':0,'new_bootstrap_runs':0,
 'source_recorder_ids':{'Stage1':sa['recorder_id'],'SOXX':sb['recorder_id']},
 'sessions':944,'rolling126_windows':819,'relative_wealth_0bp':ratio,
 'rolling126_positive_fraction_0bp':float(computed_rolling.gt(0).mean()),
 'complete_year_positive_fraction_0bp':float(complete.relative_wealth.gt(1).mean()),
 'sampled_max_drawdown_pct_0bp':float(base.sampled_point_max_drawdown_pct),
 'full_minute_risk_qualified':bool(base.fresh_fullminute_risk_qualified),
 'regime_decline_relative_log':float(regimes.set_index('regime').loc['decline','relative_log']),
 'regime_decline_sessions':int(regimes.set_index('regime').loc['decline','sessions']),
 'limitations':'Regimes are noncontiguous descriptive groups; cash and risk are endogenous account states. No exact cash/SOXL accounting decomposition is claimed here because the feature frame and raw execution price planes differ. The complete historical path is reused; no independent forward evidence.',
 'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':[],'source_recorder_authority':True}
}
annual.to_csv(card/'annual_summary.csv',index=False)
quarters.to_csv(card/'quarterly_summary.csv',index=False)
regimes.to_csv(card/'regime_summary.csv',index=False)
(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
print(json.dumps(summary,ensure_ascii=False,indent=2))

## 中文结论

只读6个已封存Qlib Recorder对象，核对944个区间、16个季度及819个连续126日窗口；0拟合、0新账户。Stage1完整链0bp额外滑点但仍扣Tiered费用，相对SOXX期末净财富比0.6860；可用点最大回撤29.49%，完整分钟风险未认证。两个完整年度仅1年正超额，126日正超额31.14%。

固定的决策前双跌状态204日相对对数贡献−0.5171，平均现金61.48%、SOXL6.48%；上行状态407日贡献+0.1016。2023Q4与2026Q3分别相对−10.36%和−15.61%，2026Q2相对+16.05%。所有状态、年度和季度均保留；非连续状态不构成可交易子策略。

决定：diagnostic_only_Stage1_0bp_not_promoted。现金、风险状态和持仓是自身净值递推的结果，不能由分组推断放松风控的因果收益。特征帧与原始股数账户价格面不同，本卡不做未对账的精确现金/SOXL会计分解。下一步核对新低扩散和下跌集中度的决策时点、缺失、来源与Stage1已有输入差异，随后冻结匹配信息实验；2/5bp与独立前向门仍未满足。